In [1]:
# import packages
import csv
import numpy as np
import pandas as pd

In [2]:
#Daftar file dan metadata
files = [
    ("sma_2021.csv", "SMA", 2021),
    ("sma_2022.csv", "SMA", 2022),
    ("sma_2023.csv", "SMA", 2023),
    ("sma_2024.csv", "SMA", 2024),
]

In [3]:
# List untuk menyimpan data
df_all = []

In [4]:
# Loop untuk membaca dan memproses tiap file
for file, jenjang, tahun in files:
    try:
        # Path ke file
        file_path = f"/content/sample_data/dataset_sma/{file}"

        # Baca CSV
        df = pd.read_csv(file_path)

        # Normalisasi nama kolom (ubah jika nama kolom berbeda)
        df = df.rename(columns={
            "Laki-laki (L) - Tingkat - X": "L_X",
            "Laki-laki (L) - Tingkat - XI": "L_XI",
            "Laki-laki (L) - Tingkat - XII": "L_XII",
            "Laki-laki (L) - Subjml": "L_Jumlah",
            "Perempuan (P) - Tingkat - X": "P_X",
            "Perempuan (P) - Tingkat - XI": "P_XI",
            "Perempuan (P) - Tingkat - XII": "P_XII",
            "Perempuan (P) - Subjml": "P_Jumlah",
        })

        # Tambah metadata Jenjang dan Tahun
        df["Jenjang"] = jenjang
        df["Tahun"] = tahun

        # Ubah ke format long
        df_long = pd.melt(
            df,
            id_vars=["Kecamatan", "Jenjang", "Tahun"],
            value_vars=["L_X", "L_XI", "L_XII", "L_Jumlah", "P_X", "P_XI", "P_XII", "P_Jumlah"],
            var_name="Kategori",
            value_name="Jumlah"
        )

        # Tambahkan kolom Jenis Kelamin dan Tingkat
        def extract_gender(k):
            return "Laki-laki" if k.startswith("L") else "Perempuan"

        def extract_grade(k):
            if "_XII" in k:
                return "XII"
            elif "_XI" in k:
                return "XI"
            elif "_X" in k:
                return "X"
            else:
                return "Total"

        df_long["Jenis Kelamin"] = df_long["Kategori"].apply(extract_gender)
        df_long["Tingkat"] = df_long["Kategori"].apply(extract_grade)

        # Hapus kolom kategori
        df_long = df_long.drop(columns="Kategori")

        # Simpan
        df_all.append(df_long)

    except Exception as e:
        print(f"Gagal memproses file {file}: {e}")

In [5]:
# Gabungkan semua jadi satu DataFrame besar
df_final = pd.concat(df_all, ignore_index=True)

In [6]:
# Tampilkan hasil
print(df_final.head())

                   Kecamatan Jenjang  Tahun  Jumlah Jenis Kelamin Tingkat
0    Kec. Teluk Betung Barat     SMA   2021       0     Laki-laki       X
1  Kec. Teluk Betung Selatan     SMA   2021       0     Laki-laki       X
2               Kec. Panjang     SMA   2021       0     Laki-laki       X
3  Kec. Tanjung Karang Timur     SMA   2021       6     Laki-laki       X
4    Kec. Teluk Betung Utara     SMA   2021       0     Laki-laki       X


In [7]:
# Simpan ke CSV
df_final.to_csv("dataset_sma.csv", index=False)